In [3]:
import pandas as pd
import importlib
import sys
import os

sys.path.append(
    os.path.abspath("../data_prepration")
)
import load_data
importlib.reload(load_data)

<module 'load_data' from 'e:\\DataAnalysis\\_Projects\\proj_17\\product_feature_performance_analysis\\data_prepration\\load_data.py'>

### **orders table duplicates check**

In [4]:
orders = load_data.load_orders()
dups_count = orders.duplicated().sum()
dups_count

np.int64(0)

### **orders table missing value check**

In [5]:
missing_value_count = orders.isna().sum()
missing_value_count

order_id          0
user_id           0
session_id        0
order_date        0
payment_method    0
order_status      0
total_amount      0
dtype: int64

### **orders table primary uniqueness check**

In [4]:
orders["order_id"].is_unique

True

### **orders orphan records by user_id**

In [5]:
users = load_data.load_users()
orphan_orders_by_user = (
    ~orders["user_id"].isin(users["user_id"])
).sum()
orphan_orders_by_user

np.int64(0)

### **orders orphan records by session_id**

In [12]:
sessions = load_data.load_sessions(columns=["session_id"])
orphan_orders_by_session = (
    ~orders["session_id"].isin(sessions["session_id"])
).sum()
orphan_orders_by_session

np.int64(0)

### **total orders, unique users and unique sessions metrics**

In [13]:
orders_overview = pd.Series({
    "Total Orders": orders["order_id"].nunique(),
    "Unique Users": orders["user_id"].nunique(),
    "Unique Sessions": orders["session_id"].nunique()
})
orders_overview

Total Orders       420000
Unique Users       189388
Unique Sessions    420000
dtype: int64

### **Order Status Distribution**

In [15]:
orders_by_status = (
    orders
        .groupby("order_status")
        .agg(
            orders_count = ("order_id", "count")
        )
)
orders_by_status["percentage"] = (orders_by_status["orders_count"] / orders_by_status["orders_count"].sum()).round(4)
orders_by_status.to_csv("../../output/orders_by_status.csv")
orders_by_status

,orders_count,percentage
order_status,,
Cancelled,20981,0.0500
Completed,386176,0.9195
Returned,12843,0.0306


### **Completed Orders by Unique Users**


In [22]:
completed_order_users = (
    orders.loc[orders["order_status"] == "Completed", "user_id"].nunique()
)
completed_order_users

182462

### **Completed Orders by Session**

In [25]:
completed_order_session = (
    orders.loc[orders["order_status"] == "Completed", "session_id"].nunique()
)
completed_order_session

386176

### **Payment Method Distribution**

In [7]:
orders_by_payment_method = (
    orders
        .groupby("payment_method")
        .agg(
            orders_count = ("order_id", "count")
        )
        .sort_values("orders_count", ascending = False)
)
orders_by_payment_method["percentage"] = (orders_by_payment_method["orders_count"] / orders_by_payment_method["orders_count"].sum()).round(4)
orders_by_payment_method.to_csv("../../output/orders_by_payment_method.csv")
orders_by_payment_method

,orders_count,percentage
payment_method,,
Credit Card,147078,0.3502
Cash,126039,0.3001
Wallet,83964,0.1999
BNPL,41892,0.0997
Bank Transfer,21027,0.0501


### **Completed Orders Monthly Distribution**

In [10]:
completed_orders_monthly_distribution = (
    orders
        .loc[orders["order_status"] == "Completed", ["order_id", "order_date"]]
        .assign(
            Year = lambda df: df["order_date"].dt.year,
            Month_Number = lambda df: df["order_date"].dt.month,
            Month_Name = lambda df: df["order_date"].dt.month_name(),
            Month_Name_Short = lambda df: df["order_date"].dt.strftime("%b"),
            Date_Label = lambda df: df["order_date"].dt.strftime("%b-%y")
        )
        .groupby(["Year", "Month_Number", "Month_Name", "Month_Name_Short", "Date_Label"])
        .agg(
            orders_count = ("order_id", "count")
        )
)
completed_orders_monthly_distribution["percentage"] = (completed_orders_monthly_distribution["orders_count"] / completed_orders_monthly_distribution["orders_count"].sum()).round(4)
completed_orders_monthly_distribution.to_csv("../../output/completed_orders_monthly_distribution.csv")
completed_orders_monthly_distribution

orders_count  \
Year Month_Number Month_Name Month_Name_Short Date_Label                 
2025 1            January    Jan              Jan-25                50   
     2            February   Feb              Feb-25               259   
     3            March      Mar              Mar-25               920   
     4            April      Apr              Apr-25              1814   
     5            May        May              May-25              3602   
     6            June       Jun              Jun-25              6025   
     7            July       Jul              Jul-25             10104   
     8            August     Aug              Aug-25             16082   
     9            September  Sep              Sep-25             24598   
     10           October    Oct              Oct-25             41108   
     11           November   Nov              Nov-25             71423   
     12           December   Dec              Dec-25            210191   

                                                          percentage  
Year Month_Number Month_Name Month_Name_Short Date_Label              
2025 1            January    Jan              Jan-25          0.0001  
     2            February   Feb              Feb-25          0.0007  
     3            March      Mar              Mar-25          0.0024  
     4            April      Apr              Apr-25          0.0047  
     5            May        May              May-25          0.0093  
     6            June       Jun              Jun-25          0.0156  
     7            July       Jul              Jul-25          0.0262  
     8            August     Aug              Aug-25          0.0416  
     9            September  Sep              Sep-25          0.0637  
     10           October    Oct              Oct-25          0.1064  
     11           November   Nov              Nov-25          0.1849  
     12           December   Dec              Dec-25          0.5443

### **Total Amount Summary Statistics**

In [4]:
total_amount_summary_stats = pd.Series({
    "Minimum": orders["total_amount"].min(),
    "Maximum": orders["total_amount"].max(),
    "Average": orders["total_amount"].mean(),
    "Q1": orders["total_amount"].quantile(0.25),
    "Median": orders["total_amount"].median(),
    "Q3": orders["total_amount"].quantile(0.75),
    "IQR": orders["total_amount"].quantile(0.75) - orders["total_amount"].quantile(0.25),
    "Standard Deviation": orders["total_amount"].std(),
    "Lower Bound": orders["total_amount"].quantile(0.25) - (1.5 * (orders["total_amount"].quantile(0.75) - orders["total_amount"].quantile(0.25))),
    "Upper Bound": orders["total_amount"].quantile(0.75) + (1.5 * (orders["total_amount"].quantile(0.75) - orders["total_amount"].quantile(0.25))),
    "Coefficient of Variation (CV)": orders["total_amount"].std() / orders["total_amount"].mean()
}).round(2)

total_amount_summary_stats

Minimum                              4.36
Maximum                          18456.98
Average                            455.29
Q1                                 192.75
Median                             330.47
Q3                                 566.25
IQR                                373.50
Standard Deviation                 432.94
Lower Bound                       -367.50
Upper Bound                       1126.51
Coefficient of Variation (CV)        0.95
dtype: float64

### **Completed Orders Distribution by Total Amount Group**

In [11]:
completed_orders = orders[orders["order_status"] == "Completed"].copy()
completed_orders["total_amount_group"] = pd.cut(
    completed_orders["total_amount"],
    bins=[0, 50, 150, 250, 450, 600, 750, 900, 1100, float("inf")],
    labels=[
        "< 50",
        "50 - 150",
        "150 - 250",
        "250 - 450",
        "450 - 600",
        "600 - 750",
        "750 - 900",
        "900 - 1,100",
        "1100+"
    ]
)

total_amount_distribution = (
    completed_orders
        .groupby("total_amount_group", observed=True)
        .agg(
            orders_count = ("order_id", "count")
        )
)

total_amount_distribution["completed orders count"] = total_amount_distribution["orders_count"].sum()
total_amount_distribution["percentage"] = (total_amount_distribution["orders_count"] / total_amount_distribution["completed orders count"]).round(4)
total_amount_distribution.to_csv("../../output/total_amount_distribution.csv")
total_amount_distribution

,orders_count,completed orders count,percentage
total_amount_group,,,
< 50,3479,386176,0.0090
50 - 150,59068,386176,0.1530
150 - 250,77660,386176,0.2011
250 - 450,111074,386176,0.2876
450 - 600,46937,386176,0.1215
600 - 750,28987,386176,0.0751
750 - 900,18427,386176,0.0477
"900 - 1,100",14838,386176,0.0384
1100+,25706,386176,0.0666
